In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import transformers
transformers.__version__

'5.0.0'

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("deberta-xsmall-full/config.json"))
path = matches[0].parent
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path, dtype=torch.float32)
print("OK")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

OK


In [4]:
import json

def to_text(s):
    parts = json.loads(s)
    return " ".join(x if x is not None else "" for x in parts)

def preprocess_test(example):
    p = tokenizer(to_text(example["prompt"]),     add_special_tokens=False)["input_ids"][:128]
    a = tokenizer(to_text(example["response_a"]), add_special_tokens=False)["input_ids"][:180]
    b = tokenizer(to_text(example["response_b"]), add_special_tokens=False)["input_ids"][:180]
    sep = tokenizer.sep_token_id
    cls = tokenizer.cls_token_id
    ids = [cls] + p + [sep] + a + [sep] + b + [sep]
    return {"input_ids": ids, "attention_mask": [1]*len(ids)}

In [5]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

test_df = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")
test_ds = Dataset.from_pandas(test_df)
tokenized_ds = test_ds.map(preprocess_test, remove_columns=test_ds.column_names)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [6]:
test_df.head()

,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran..."
1,211333,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ..."
2,1233961,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p..."


In [7]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

collator = DataCollatorWithPadding(tokenizer)
loader = DataLoader(tokenized_ds, batch_size=64, collate_fn=collator)

In [8]:
from scipy.special import softmax

all_probs = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        logits = model(**batch).logits
        probs = softmax(logits.float().cpu().numpy(), axis=-1)
        all_probs.append(probs)

probs = np.concatenate(all_probs, axis=0)
sub = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_tie":     probs[:, 2],
})
sub.to_csv("/kaggle/working/submission.csv", index=False)

In [9]:
# sub.shape
# test_df.shape
# test_df.head()
# sub.head()
# pd.read_csv("/kaggle/working/submission.csv").head()

# id = 2
# test_df.iloc[id]["prompt"]
# test_df.iloc[id]["response_a"]
# test_df.iloc[id]["response_b"]
